# EasyVitessce Example: SpatialData-Plot with blobs dataset

## Import EasyVitessce

This import statement enables interactive plots by default.
Refer to the [EasyVitessce documentation](https://vitessce.github.io/easy_vitessce/) for how to disable interactive plotting or configure other behaviors.

In [1]:
import easy_vitessce as ev
import spatialdata
import spatialdata_plot
from anndata import AnnData
import pandas as pd
from vitessce.data_utils import sdata_points_process_columns

/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
ev.config.set({ 'data.overwrite': True })

Un-comment the line below if the notebook kernel is running on a different machine (e.g., Google Colab, Docker container, or HPC cluster)

In [3]:
# ev.config.set({ 'data.wrapper_param_suffix': '_store' })

## Read the example SpatialData object

In [4]:
sdata = spatialdata.datasets.blobs()
sdata

/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


SpatialData object
├── Images
│     ├── 'blobs_image': DataArray[cyx] (3, 512, 512)
│     └── 'blobs_multiscale_image': DataTree[cyx] (3, 512, 512), (3, 256, 256), (3, 128, 128)
├── Labels
│     ├── 'blobs_labels': DataArray[yx] (512, 512)
│     └── 'blobs_multiscale_labels': DataTree[yx] (512, 512), (256, 256), (128, 128)
├── Points
│     └── 'blobs_points': DataFrame with shape: (<Delayed>, 4) (2D points)
├── Shapes
│     ├── 'blobs_circles': GeoDataFrame shape: (5, 2) (2D shapes)
│     ├── 'blobs_multipolygons': GeoDataFrame shape: (2, 1) (2D shapes)
│     └── 'blobs_polygons': GeoDataFrame shape: (5, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (26, 3)
with coordinate systems:
    ▸ 'global', with elements:
        blobs_image (Images), blobs_multiscale_image (Images), blobs_labels (Labels), blobs_multiscale_labels (Labels), blobs_points (Points), blobs_circles (Shapes), blobs_multipolygons (Shapes), blobs_polygons (Shapes)

### Create a feature_index column for the Points table

In [5]:
ddf = sdata.points['blobs_points']
ddf

,x,y,genes,instance_id
npartitions=1,,,,
0,int64,int64,category[known],int64
199,...,...,...,...


In [6]:
print(sdata.tables['table'].var.index.tolist())

['channel_0_sum', 'channel_1_sum', 'channel_2_sum']


In [7]:
# The sdata.tables['table'].var table does not contain genes as indices; it contains channels.
# We need to create another table which has a var.index column containing the gene IDs for the points.
unique_gene_ids = ddf["genes"].unique().compute().tolist()
points_var_df = pd.DataFrame(index=unique_gene_ids, data=[], columns=[])
points_table = AnnData(var=points_var_df, obs=None, X=None)
sdata.tables['table_points'] = points_table

In [8]:
sdata.points['blobs_points'] = sdata_points_process_columns(sdata, "blobs_points", var_name_col="genes", table_name="table_points")

/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/dask/dataframe/core.py:4448: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map or apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, meta=('genes', 'category'))

  warnings.warn(meta_warning(meta))
/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/spatialdata/_core/_elements.py:115: UserWarning: Key `blobs_points` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


In [13]:
sdata.write('data/blobs_with_feature_index.sdata.zarr')

INFO     The Zarr backing store has been changed from data/1953eb5a-d483-4694-ad80-4c247689706a.sdata.zarr the new 
         file path: data/blobs_with_feature_index.sdata.zarr                                                       


## Plot the data

In [15]:
sdata.pl.render_images("blobs_image").pl.render_labels("blobs_labels").pl.show()

VitessceWidget(js_dev_mode=True, uid='5508')

### With color parameter and with points

In [16]:
sdata.pl.render_images("blobs_image").pl.render_labels("blobs_labels", color="channel_0_sum", table_name="table").pl.render_points("blobs_points").pl.show()

VitessceWidget(js_dev_mode=True, uid='b28f')

## Disable EasyVitessce

In [ ]:
ev.disable_plots(["spatialdata-plot"])

## Generate a static plot using the same code

In [ ]:
sdata.pl.render_images("blobs_image").pl.render_labels("blobs_labels").pl.show()

In [ ]:
sdata.pl.render_images("blobs_image").pl.render_labels("blobs_labels", color="channel_0_sum").pl.render_points("blobs_points").pl.show()